In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
import findspark

findspark.init("/Users/i545672/spark3/spark-3.5.5-bin-hadoop3")
findspark.find()

'/Users/i545672/spark3/spark-3.5.5-bin-hadoop3'

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = (
    SparkSession
        .builder
        .appName("WindowOperationsApp")
        .master("local[4]")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.sql.adaptive.enabled", "false")
        .getOrCreate()
)

sc= spark.sparkContext
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/25 21:45:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
yellowTaxiSchema = StructType([
 StructField("VendorID", IntegerType(), True),
 StructField("tpep_pickup_datetime", TimestampType(), True),
 StructField("tpep_dropoff_datetime", TimestampType(), True),
 StructField("passenger_count", DoubleType(), True),
 StructField("trip_distance", DoubleType(), True),
 StructField("RatecodeID", DoubleType(), True),
 StructField("store_and_fwd_flag", StringType(), True),
 StructField("PickUpLocationID", IntegerType(), True),
 StructField("DOLocationID", IntegerType(), True),
 StructField("payment_type", IntegerType(), True),
 StructField("fare_amount", DoubleType(), True),
 StructField("extra", DoubleType(), True),
 StructField("mta_tax", DoubleType(), True),
 StructField("tip_amount", DoubleType(), True),
 StructField("tolls_amount", DoubleType(), True),
 StructField("improvement_surcharge", DoubleType(), True),
 StructField("total_amount", DoubleType(), True),
 StructField("congestion_surcharge", DoubleType(), True),
 StructField("airport_fee", DoubleType(), True),
])
yellowTaxisDF = spark.read.option("header", "true").schema(yellowTaxiSchema).csv(
    "./Files/YellowTaxis_202210.csv"
)
yellowTaxisDF.createOrReplaceTempView("YellowTaxis")
yellowTaxisDF.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PickUpLocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [7]:
taxiZonesSchema = 'PickUpLocationID INT, Borough STRING, Zone STRING, ServiceZone STRING'

taxiZonesDF = spark.read.schema(taxiZonesSchema).csv(
    "./Files/TaxiZones.csv"
)

taxiZonesDF.createOrReplaceTempView("TaxiZones")

taxiZonesDF.printSchema()

root
 |-- PickUpLocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- ServiceZone: string (nullable = true)



In [8]:
# Get rides for each Borough
taxiRidesDF = spark.sql(" SELECT tz.Borough, COUNT(*) AS RideCount FROM TaxiZones tz INNER JOIN YellowTaxis yt ON tz.PickUpLocationID = yt.PickUpLocationID GROUP BY tz.Borough")
taxiRidesDF.createOrReplaceTempView("TaxiRides")
taxiRidesDF.orderBy("Borough").show()

25/05/25 21:46:15 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: PULocationID
 Schema: PickUpLocationID
Expected: PickUpLocationID but found: PULocationID
CSV file: file:///Users/i545672/SAPDevelop/Spark/Files/YellowTaxis_202210.csv


+-------------+---------+
|      Borough|RideCount|
+-------------+---------+
|        Bronx|     4511|
|     Brooklyn|    28089|
|          EWR|     1157|
|    Manhattan|  3250695|
|       Queens|   333922|
|Staten Island|      303|
|      Unknown|    56735|
+-------------+---------+



In [14]:
# Calculate taxi rides across all Boroughs by creating a window over the entire dataset and adding total rides (across all boroughs) against each row
taxiRidesWindowDF = spark.sql("""
    SELECT 
        *, 
        SUM(RideCount) OVER () AS TotalRideCount
    FROM TaxiRides
""")
taxiRidesWindowDF.orderBy("Borough").show()
taxiRidesWindowDF.createOrReplaceTempView("TaxiRidesWindow")

25/05/25 21:54:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/05/25 21:54:36 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: PULocationID
 Schema: PickUpLocationID
Expected: PickUpLocationID but found: PULocationID
CSV file: file:///Users/i545672/SAPDevelop/Spark/Files/YellowTaxis_202210.csv


+-------------+---------+--------------+
|      Borough|RideCount|TotalRideCount|
+-------------+---------+--------------+
|        Bronx|     4511|       3675412|
|     Brooklyn|    28089|       3675412|
|          EWR|     1157|       3675412|
|    Manhattan|  3250695|       3675412|
|       Queens|   333922|       3675412|
|Staten Island|      303|       3675412|
|      Unknown|    56735|       3675412|
+-------------+---------+--------------+



In [16]:
# Find the share of each borough in the total rides by dividing the ride count of each borough by the total ride count and multiplying by 100 to get a percentage
taxiRidesShareDF = spark.sql("""
    SELECT 
        * ,
        ROUND( (RideCount * 100) / TotalRideCount, 2) AS RidesSharePercent
    FROM TaxiRidesWindow
    ORDER BY Borough
""").show()

25/05/25 21:55:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/05/25 21:55:10 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: PULocationID
 Schema: PickUpLocationID
Expected: PickUpLocationID but found: PULocationID
CSV file: file:///Users/i545672/SAPDevelop/Spark/Files/YellowTaxis_202210.csv


+-------------+---------+--------------+-----------------+
|      Borough|RideCount|TotalRideCount|RidesSharePercent|
+-------------+---------+--------------+-----------------+
|        Bronx|     4511|       3675412|             0.12|
|     Brooklyn|    28089|       3675412|             0.76|
|          EWR|     1157|       3675412|             0.03|
|    Manhattan|  3250695|       3675412|            88.44|
|       Queens|   333922|       3675412|             9.09|
|Staten Island|      303|       3675412|             0.01|
|      Unknown|    56735|       3675412|             1.54|
+-------------+---------+--------------+-----------------+



In [ ]:
# Find share of each zone in terms of rides within their borough
taxiRidesDF = spark.sql("""
    SELECT 
        tz.Borough, 
        tz.Zone, 
        COUNT(*) AS RideCount
    FROM TaxiZones tz 
    INNER JOIN YellowTaxis yt ON tz.PickUpLocationID = yt.PickUpLocationID
    GROUP BY tz.Borough, tz.Zone
""")
taxiRidesDF.orderBy("Borough", "Zone").show()
taxiRidesDF.createOrReplaceTempView("TaxiRidesByZone")

25/05/25 21:58:15 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: PULocationID
 Schema: PickUpLocationID
Expected: PickUpLocationID but found: PULocationID
CSV file: file:///Users/i545672/SAPDevelop/Spark/Files/YellowTaxis_202210.csv


+-------+--------------------+---------+
|Borough|                Zone|RideCount|
+-------+--------------------+---------+
|  Bronx|Allerton/Pelham G...|       51|
|  Bronx|        Bedford Park|       92|
|  Bronx|             Belmont|       59|
|  Bronx|          Bronx Park|       22|
|  Bronx|           Bronxdale|       48|
|  Bronx|         City Island|       11|
|  Bronx|  Claremont/Bathgate|       98|
|  Bronx|          Co-Op City|      200|
|  Bronx|        Country Club|        7|
|  Bronx|        Crotona Park|        2|
|  Bronx|   Crotona Park East|       45|
|  Bronx|East Concourse/Co...|      180|
|  Bronx|        East Tremont|      113|
|  Bronx|         Eastchester|       73|
|  Bronx|       Fordham South|       52|
|  Bronx|          Highbridge|      158|
|  Bronx|         Hunts Point|       67|
|  Bronx| Kingsbridge Heights|      116|
|  Bronx|            Longwood|       41|
|  Bronx|       Melrose South|      173|
+-------+--------------------+---------+
only showing top

In [19]:
# Calculate total rides across each zone and borough by creating a window partitioned by borough over entire table
taxiRidesByZoneWindowDF = spark.sql("""
    SELECT 
        *, 
        SUM(RideCount) OVER (PARTITION BY Borough) AS TotalRideCountByBorough
    FROM TaxiRidesByZone
""")
taxiRidesByZoneWindowDF.orderBy("Borough", "Zone").show()
taxiRidesByZoneWindowDF.createOrReplaceTempView("TaxiRidesByZoneWindow")

25/05/25 22:03:56 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: PULocationID
 Schema: PickUpLocationID
Expected: PickUpLocationID but found: PULocationID
CSV file: file:///Users/i545672/SAPDevelop/Spark/Files/YellowTaxis_202210.csv


+-------+--------------------+---------+-----------------------+
|Borough|                Zone|RideCount|TotalRideCountByBorough|
+-------+--------------------+---------+-----------------------+
|  Bronx|Allerton/Pelham G...|       51|                   4511|
|  Bronx|        Bedford Park|       92|                   4511|
|  Bronx|             Belmont|       59|                   4511|
|  Bronx|          Bronx Park|       22|                   4511|
|  Bronx|           Bronxdale|       48|                   4511|
|  Bronx|         City Island|       11|                   4511|
|  Bronx|  Claremont/Bathgate|       98|                   4511|
|  Bronx|          Co-Op City|      200|                   4511|
|  Bronx|        Country Club|        7|                   4511|
|  Bronx|        Crotona Park|        2|                   4511|
|  Bronx|   Crotona Park East|       45|                   4511|
|  Bronx|East Concourse/Co...|      180|                   4511|
|  Bronx|        East Tre

In [20]:
# Find share each zone in terms of rides within their borough by dividing the ride count of each zone by the total ride count of the borough and multiplying by 100 to get a percentage
taxiRidesByZoneShareDF = spark.sql("""
    SELECT 
        *,
        ROUND( (RideCount * 100) / TotalRideCountByBorough, 2) AS RidesSharePercent
    FROM TaxiRidesByZoneWindow
    ORDER BY Borough, Zone
""").show()

25/05/25 22:06:14 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: PULocationID
 Schema: PickUpLocationID
Expected: PickUpLocationID but found: PULocationID
CSV file: file:///Users/i545672/SAPDevelop/Spark/Files/YellowTaxis_202210.csv


+-------+--------------------+---------+-----------------------+-----------------+
|Borough|                Zone|RideCount|TotalRideCountByBorough|RidesSharePercent|
+-------+--------------------+---------+-----------------------+-----------------+
|  Bronx|Allerton/Pelham G...|       51|                   4511|             1.13|
|  Bronx|        Bedford Park|       92|                   4511|             2.04|
|  Bronx|             Belmont|       59|                   4511|             1.31|
|  Bronx|          Bronx Park|       22|                   4511|             0.49|
|  Bronx|           Bronxdale|       48|                   4511|             1.06|
|  Bronx|         City Island|       11|                   4511|             0.24|
|  Bronx|  Claremont/Bathgate|       98|                   4511|             2.17|
|  Bronx|          Co-Op City|      200|                   4511|             4.43|
|  Bronx|        Country Club|        7|                   4511|             0.16|
|  B